# 手势识别（机械臂主机内直接运行）

和 `Face.ipynb` 一样：Jupyter 内核直接打开摄像头（`cv.VideoCapture(0)`），
逐帧做 MediaPipe 手势识别，画面用 `ipywidgets.Image` 实时回显，
识别结果同时打印到单元输出并写入 `gesture_commands.jsonl`。

## 前提

1. 先**停止**任何正在使用摄像头的内核（例如 `Face.ipynb` 的人脸追踪线程），
   否则 `/dev/video0` 被占用，会报 `can't open camera by index`。
2. 本目录下需要有 `gesture/`（含 `models/` 两个 `.task` 模型），本 notebook 与它同级。
3. 依赖：`opencv-python`(或系统自带 `python3-opencv`)、`numpy`、`mediapipe`。
   板子**不要**再用 `%pip install`（网络 DNS 不通），用离线包或系统自带版本。

## 运行顺序

依赖自检 → 目录/模型校验 → 摄像头自检 → 实时手势识别 → 结果汇总 →（可选）视频文件验证

In [ ]:
# 1) 依赖自检（不联网安装，只检查）
import importlib, sys
print("python:", sys.version.split()[0])

for name in ("cv2", "numpy", "mediapipe"):
    try:
        m = importlib.import_module(name)
        print(f"OK   {name}", getattr(m, "__version__", "?"))
    except Exception as e:
        print(f"MISS {name}: {e}")

try:
    import cv2
    print("cv2 路径:", cv2.__file__)
except Exception:
    pass

In [ ]:
# 2) 定位工程目录 + 校验模型
from pathlib import Path
import sys, json, time, threading

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_SRC = next((p for p in candidates if (p / "gesture").is_dir()), None)
if PROJECT_SRC is None:
    raise SystemExit("未找到 gesture 目录：请把 src/gesture 上传到本项目目录下")

sys.path.insert(0, str(PROJECT_SRC))
from gesture.gesture_command import GESTURE_CODES
from gesture.recognizer import HandRecognizer
from gesture.command_gate import CommandGate
from gesture.logger import RecognitionLogger

models = sorted(p.name for p in (PROJECT_SRC / "gesture" / "models").glob("*.task"))
print("PROJECT_SRC =", PROJECT_SRC)
print("模型文件 =", models)
print("手势映射 =", GESTURE_CODES)
assert {"gesture_recognizer.task", "hand_landmarker.task"} <= set(models), "模型不完整"

In [ ]:
# 3) 摄像头自检：与 Face.ipynb 相同的打开方式（重试 + 设分辨率）
import cv2, time
from IPython.display import display, Image

def open_camera(index=0, retries=6, width=320, height=240, fps=30):
    """带重试地打开摄像头；失败时给出可操作的错误提示。"""
    last = None
    for i in range(retries):
        cap = cv2.VideoCapture(index)
        if cap.isOpened():
            cap.set(3, width)
            cap.set(4, height)
            cap.set(5, fps)
            ok, frame = cap.read()
            if ok and frame is not None:
                return cap
            last = "打开成功但读不到画面（可能被其它进程占用）"
            cap.release()
        else:
            last = "cap.isOpened() == False"
        time.sleep(1.0)
    raise RuntimeError(
        f"无法打开摄像头 {index}（{last}）。请依次检查：\n"
        "  1) 先停止 Face.ipynb 等占用摄像头的内核/线程\n"
        "  2) ls -l /dev/video* 是否有设备\n"
        "  3) id 是否包含 video 组\n"
        "  4) 其它索引：open_camera(1) / open_camera(2)"
    )

cap = open_camera(0)
ok, frame = cap.read()
cap.release()
ok_bytes, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
display(Image(data=buf.tobytes(), width=480))
print("画面尺寸:", frame.shape)

In [ ]:
# 4) 实时手势识别：在 notebook 进程内跑 20 秒（画面实时回显）
import cv2, json, time
import ipywidgets as widgets
from IPython.display import display

OUT_FILE = PROJECT_SRC / "gesture_commands.jsonl"
LOG_FILE = PROJECT_SRC / "recognition_log.csv"
RUN_SECONDS = 20

recognizer = HandRecognizer(min_confidence=0.5, smooth_alpha=0.6)
gate = CommandGate(min_confidence=0.5, use_calibration=False)
logger = RecognitionLogger(str(LOG_FILE))
preview = widgets.Image(format="jpeg", width=480, height=360)
display(preview)

cap = open_camera(0)
records = []
start = time.time()
try:
    while time.time() - start < RUN_SECONDS:
        ok, frame = cap.read()
        if not ok:
            print("读帧失败，退出循环")
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = recognizer.process_frame(rgb)
        gid = res.gesture_id if res is not None else "NONE"
        conf = res.confidence if res is not None else 0.0
        raw_conf = res.base_conf if res is not None else 0.0

        now = time.time()
        cmd = gate.update(gid, conf, now)
        logger.log(gid, conf, cmd is not None, raw_confidence=raw_conf)

        label = f"{gid} {conf:.2f}"
        color = (0, 200, 0) if cmd is not None else (200, 200, 200)
        cv2.putText(frame, label, (8, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        if cmd is not None:
            print(">> 命令:", cmd.to_json(), flush=True)
            payload = json.loads(cmd.to_json())
            records.append(payload)
            with open(OUT_FILE, "a", encoding="utf-8") as f:
                f.write(cmd.to_json() + "\n")

        ok_jpg, jpg = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 60])
        if ok_jpg:
            preview.value = jpg.tobytes()
        time.sleep(0.02)
finally:
    cap.release()
    recognizer.close()

print(f"本次运行 {RUN_SECONDS}s，下发 {len(records)} 条手势命令")
for r in records[-10:]:
    print(r["gesture_id"], "code=", r["gesture_code"], "conf=", r["confidence"], "task_id=", r["task_id"])

In [ ]:
# 5) 历史结果汇总（读取 jsonl，确认落盘）
import json
from pathlib import Path
from collections import Counter

rows = []
if OUT_FILE.exists():
    for ln in OUT_FILE.read_text(encoding="utf-8").splitlines():
        ln = ln.strip()
        if ln.startswith("{"):
            rows.append(json.loads(ln))
print("累计命令数:", len(rows))
print("分布:", Counter(r["gesture_id"] for r in rows))
rows[-5:]

In [ ]:
# 6) 可选：没有摄像头时用视频文件验证（把 demo.mp4 放到本目录）
# import subprocess, sys
# VIDEO = str(PROJECT_SRC / "demo.mp4")
# !{sys.executable} -m gesture.app --video "{VIDEO}" --sink file --out video_commands.jsonl

## 与 Face.ipynb 的对应关系

| Face.ipynb | 本 notebook |
|---|---|
| `capture = cv.VideoCapture(0)` | `open_camera(0)`（带重试 + 320×240） |
| `imgbox.value = cv.imencode(...)` 线程回显 | `preview.value = jpg.tobytes()` 同步回显 |
| `Arm_serial_servo_write6_array(...)` 驱动舵机 | **不驱动舵机**，只输出 `GestureCommand` |

## 和机械臂联动

本 notebook 只负责识别。要让舵机动起来，由 C 侧读取 `gesture_commands.jsonl`
（或直接消费 `records`）里的 `gesture_code`，查表后经急停/限位校验再下发：

```python
GESTURE_TO_MOTION = {
    1: ("servo", 2, "up", 10),     # FIST
    2: ("servo", 2, "down", 10),   # OPEN_PALM
    3: ("home",),                  # POINT_UP
    7: ("estop",),                 # THUMBS_UP
}
```